# DATASETS PREPARATION

In [ ]:
import h5py
from sklearn.model_selection import train_test_split

In [ ]:
background_file = "../data/raw/background_for_training.h5"
background_output_dense = "../data/datasets/dense/background_dense_dataset.h5"
background_output_convolutional = "../data/datasets/convolutional/background_convolutional_dataset.h5"

signal_files = ["../data/raw/Ato4l_lepFilter_13TeV_filtered.h5",
               "../data/raw/hChToTauNu_13TeV_PU20_filtered.h5",
               "../data/raw/hToTauTau_13TeV_PU20_filtered.h5",
               "../data/raw/leptoquark_LOWMASS_lepFilter_13TeV_filtered.h5"]
signal_output_dense = ["../data/datasets/dense/Ato4l_dense_dataset.h5",
                       "../data/datasets/dense/hChToTauNu_dense_dataset.h5",
                       "../data/datasets/dense/hToTauTau_dense_dataset.h5",
                       "../data/datasets/dense/leptoquark_dense_dataset.h5"]
signal_output_convolutional = ["../data/datasets/convolutional/Ato4l_convolutional_dataset.h5",
                               "../data/datasets/convolutional/hChToTauNu_convolutional_dataset.h5",
                               "../data/datasets/convolutional/hToTauTau_convolutional_dataset.h5",
                               "../data/datasets/convolutional/leptoquark_convolutional_dataset.h5"]

test_size = 0.2
val_size = 0.2

In [ ]:
# read BACKGROUND data
with h5py.File(background_file, 'r') as f:
    background_data = f['Particles'][:,:,:-1]   # ignore label (N, 19, 4) -> (N, 19, 3)

X_train, X_test = train_test_split(background_data, test_size=test_size, shuffle=True)
X_train, X_val = train_test_split(X_train, test_size=val_size, shuffle=True)

del background_data

X_train_dense = X_train.reshape(X_train.shape[0], -1)    # (N, 19, 3) -> (N, 57) for dense
X_test_dense = X_test.reshape(X_test.shape[0], -1)
X_val_dense = X_val.reshape(X_val.shape[0], -1)

with h5py.File(background_output_dense, 'w') as h5fd:
    h5fd.create_dataset('X_train', data=X_train_dense)
    h5fd.create_dataset('X_test', data=X_test_dense)
    h5fd.create_dataset('X_val', data=X_val_dense)

del X_train_dense, X_test_dense, X_val_dense

with h5py.File(background_output_convolutional, 'w') as h5fc:
    h5fc.create_dataset('X_train', data=X_train)
    h5fc.create_dataset('X_test', data=X_test)
    h5fc.create_dataset('X_val', data=X_val)

del X_train, X_test, X_val

In [ ]:
# read SIGNAL data
assert len(signal_files) == len(signal_output_dense) == len(signal_output_convolutional), "Mismatch in number of signal files and output files"

for signal_file, output_file_dense, output_file_convolutional in zip(signal_files, signal_output_dense, signal_output_convolutional):

    with h5py.File(signal_file, 'r') as f:
        signal_data = f['Particles'][:,:,:-1]

    signal_data_dense = signal_data.reshape(signal_data.shape[0], -1)

    with h5py.File(output_file_dense, 'w') as h5fd:
        h5fd.create_dataset('Data', data=signal_data_dense)

    with h5py.File(output_file_convolutional, 'w') as h5fc:
        h5fc.create_dataset('Data', data=signal_data)

    del signal_data, signal_data_dense